# Demonstration 

## Setup

First, it is necessary to import packages. A `Munch` is a `Dict` that acts like a structure, so we can use dot notation. We'll use this to demonstrate different functions and keep the results organized.

In [1]:
import numpy as np
import astropy.units as u
from munch import Munch

import tellurion.core as tell
import tellurion.ork as tork

Next we define a specific epoch time `newyear` and a sequence of time steps every 5 minutes for one hour `prop5m1h`

In [3]:
newyear = tell.abstime('2025-01-01T00:00:00')
prop5m1h = np.linspace(5.0*u.minute, 60.0*u.minute, 12) # Step every 5 minutes for an hour

## State and elements


Create the tree structure `demoa` to hold values; `demoa.init` will hold initial values.

In [4]:
demoa = Munch()
demoa.init = Munch()

Create a position, velocity and time, representing the __[orbital state vector](https://en.wikipedia.org/wiki/Orbital_state_vectors)__

In [5]:
demoa.init.pos = [5740.13268349, 3314.06715   ,    0.]
demoa.init.vel = [-2.75082684,  4.76457184,  5.50165367]
demoa.init.pvt = tell.pvt((demoa.init.pos, demoa.init.vel, newyear))  # A tuple (Quantity, Time)
demoa.init.pvt

PVT(pv=<Quantity ([5740.13268349, 3314.06715   ,    0.        ], [-2.75082684,  4.76457184,  5.50165367]) (km, km / s)>, time=<Time object: scale='utc' format='isot' value=2025-01-01T00:00:00.000>, aux={})

Transform this into a Kepler element set and display it as a dictionary of dimensioned quantities

In [6]:
demoa.init.kep = tork.kepler(demoa.init.pvt)  # Convert PVT to Kepler elements
tell.splitsq(demoa.init.kep[0])  # Easier to read Kepler elements

{'inc': <Quantity 44.99999999 deg>,
 'argper': <Quantity 5.42238634e-06 deg>,
 'raan': <Quantity 30. deg>,
 'ecc': <Quantity 0.00663034>,
 'sma': <Quantity 6672.37441089 km>,
 'ma': <Quantity -5.3508378e-06 deg>}

## Two-body propagation

Propagation results will be saved in `demoa.prop`.

In [7]:
demoa.prop = Munch()

Make the propagation generator, which is necessary to create an ephemeris. A maximum time must be specified; we will make it 1 day.

In [9]:
demoa.prop.gen = tork.prepare(demoa.init.pvt, 1*u.day)

Propagate for 1 hour in 5 minute steps, and include the initial state in the ephemeris table; note the middle (`Y`) value for the position and velocity vectors has been elided, but it is present 

In [10]:
demoa.prop.ephem = tork.propagate(demoa.prop.gen, prop5m1h, True)
demoa.prop.ephem

time,from epoch,elapsed,position,velocity
,,,km,km / s
Time,object,str4,float64[3],float64[3]
2025-01-01T00:00:00.000,0.0s,,5740.386 .. -0.508,-2.750099 .. 5.501650
2025-01-01T00:05:00.000,5min,5min,4582.268 .. 1616.348,-4.890869 .. 5.166623
2025-01-01T00:10:00.000,10min,5min,2866.094 .. 3036.456,-6.432025 .. 4.204201
2025-01-01T00:15:00.000,15min,5min,801.930 .. 4088.186,-7.187859 .. 2.737021
2025-01-01T00:20:00.000,20min,5min,-1359.307 .. 4646.465,-7.074231 .. 0.948990
2025-01-01T00:25:00.000,25min,5min,-3357.767 .. 4647.391,-6.115585 .. -0.940710
2025-01-01T00:30:00.000,30min,5min,-4956.379 .. 4094.522,-4.436880 .. -2.706269
2025-01-01T00:35:00.000,35min,5min,-5968.288 .. 3056.748,-2.243920 .. -4.142076


Find the altitudes of perigee and apogee; because a two-body propagator is used, they only change due to the slow rotation of the coordinate system.

In [11]:
demoa.prop.altperapo = tork.tselements(demoa.prop.ephem, ["altper","altapo"])
demoa.prop.altperapo

time,altper,altapo
,km,km
Time,float64,float64
2025-01-01T00:00:00.000,249.99757463709545,338.45922373693065
2025-01-01T00:05:00.000,249.99757458888087,338.4592237739973
2025-01-01T00:10:00.000,249.99743532263489,338.45932169691844
2025-01-01T00:15:00.000,249.99728994078097,338.45932122097815
2025-01-01T00:20:00.000,249.9969076300608,338.4595197558459
2025-01-01T00:25:00.000,249.99667646081465,338.4594314180054
2025-01-01T00:30:00.000,249.99630308960752,338.45950143718534
2025-01-01T00:35:00.000,249.99610145957118,338.4592720328691


## Data at single points

This section shows how to get single-point propagation data. In the first computation, propagate to 12 hours with the previously-defined generator, which is within the maximum allowable time of 1 day for this generator.

Find the state at step 5. The use of `tell.splitsq` in the last step is to make the elements result easier to read; access to components, e.g. `...['raan']`, is the same whether `...` is `demoa.prop.gen.kepler()[0]` or the split form.

In [19]:
pvt5 = tell.pvt(demoa.prop.ephem[5])
tell.splitsq(pvt5.pv)

{'position': <Quantity [-3357.62415675,  3427.83804991,  4647.40690831] km>,
 'velocity': <Quantity [-6.11568731, -4.61697531, -0.94057425] km / s>}